Deep neural networks suffer from internal covariate shift — the distribution of inputs to each layer changes during training, making optimization hard. Both BatchNorm and LayerNorm normalize activations to stabilize training.

The difference is WHICH dimensions they normalize over.
 For a transformer, activations are typically:
 [B, S, D]
  B = batch size
 S = sequence length
  D = hidden dimension (d_model)

In [ ]:
# What LayerNorm Does
# LayerNorm normalizes each token's 
# feature vector independently. For each (b, s) position, it normalizes across the D features.


import torch 
import torch.nn as nn
class LayerNorm(nn.Module):

    def __init__(self, normalised_shape: int, eps: float = 1e-5):
        super().__init__()
        self.eps=eps
        self.gamma = nn.Parameter(torch.ones(normalised_shape))
        self.betta= nn.Parameter(torch.zeros(normalised_shape))
    
    def forward(self,x):

        # x--> [B,S,N]
        # y--> [B,S,N]

        mean_x=torch.mean(x,dim=-1,keepdim=True) # (B,S,1)

        var_x=torch.var(x,dim=-1,keepdim=True,unbiased=False) # (B,S,1)

        x_norm = (x-mean_x)/ torch.sqrt(var_x+self.eps)

        return x_norm*self.gamma + self.betta 


In [ ]:
# torch.randn(B,S,N)
x=torch.randn(2,4,8)

y=LayerNorm(normalised_shape=8,eps=1e-3)

out=y(x)
out

2️⃣ BatchNorm — The Image Standard
What BatchNorm Does
BatchNorm normalizes each feature independently across the batch. For each feature d, it computes statistics over all (b, s) positions.

In [21]:
# For each feature d:
#     Compute mean over B*S positions
#     Compute variance over B*S positions
#     Normalize: (x - mean) / sqrt(var + eps)
#     Scale and shift: gamma * normalized + beta

class BatchNorm1d(nn.Module):

    def __init__(self,num_features: int, eps: int,momentum:int):
        super().__init__()

        self.num_features=num_features
        self.eps=eps
        self.momentum=momentum

        self.gamma=nn.Parameter(torch.ones(num_features))
        self.betta=nn.Parameter(torch.zeros(num_features))

        self.register_buffer('running_mean',torch.zeros(num_features))
        self.register_buffer('running_var',torch.ones(num_features))

        self.num_batches_tracked =0


    def forward(self,x):

        running_dim=[0,1]

        if self.training:
            x_mean=torch.mean(x,dim=running_dim,keepdim=True)
            x_var=torch.var(x,dim=running_dim,unbiased=True)
# [B, S, D] - [D] = [B, S, D]
            x_norm= (x-x_mean)/ torch.sqrt(x_var+self.eps)

            with torch.no_grad():
                self.running_mean = (1 - self.momentum) * self.running_mean + self.momentum * x_mean
                self.running_var = (1 - self.momentum) * self.running_var + self.momentum * x_var
                self.num_batches_tracked += 1


        else:
            # [B, S, D] - [D] = [B, S, D]
            x_norm= (x-self.running_mean)/ torch.sqrt(self.running_var+self.eps)


        return self.gamma*x_norm + self.betta



In [ ]:
B,S,D=2,4,8

layer=BatchNorm1d(num_features=D,eps=1e-5,momentum=0.1)

x=torch.randn(B,S,D)

out=layer(x)
out

## RMS NORM why LLMs use it

In [ ]:
# LLaMA, Mistral, and most modern LLMs use RMSNorm instead of LayerNorm. It's simpler and faster:

class RMSNorm(nn.Module):

    def __init__(self,eps: int= 1e-4,dim: int= 8):
        super().__init__()

        self.eps=eps
        self.dim=dim
        self.weight= nn.Parameter(torch.ones(self.dim))

    def forward(self,x):

        variance = torch.mean(x.pow(2),dim=-1,keepdim=True)
           # Scale the input by its root mean square and apply the learnable weight
        # x * rsqrt(variance + eps) is computationally faster than x / sqrt(variance + eps)

        # torch.rsqrt: This calculates the reciprocal root mean square 
        # (\(1 / \sqrt{x}\)) directly. It is much faster and more stable than doing torch.sqrt() followed by division.

        return x*torch.rsqrt(variance+self.eps)*self.weight

